In [4]:
%pip install weasyprint 

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------------ --- 2.1/2.3 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 10.2 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ------------- -------------------------- 2.4/7.1 MB 11.8 MB/s eta 0:00:01
   -------------------------- ------------- 4.7/7.1 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  7.1/7.1 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 10.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 10.2 MB/s  0:00:00

   ------ ---------------------------------  2/13 [zopfli]
   --------- ------------------------------  3/13 [tinyhtml5]
   ------------ ---------------------------  4/13 [tinycss2]
   --------------- ------------------------  5/13 [Pyphen]
   ------------------

In [ ]:
from pydantic import BaseModel, EmailStr, HttpUrl, Field
from typing import Optional
from datetime import date

def alarm():
    import nava, time
    from nava import stop
    sound_id = nava.play('../alarm.wav', async_mode=True)
    time.sleep(20)
    stop(sound_id)

    
class ContactInfo(BaseModel):
    full_name: str
    job_title: str
    email: str
    phone: str
    location: str
    linkedin_url: Optional[str] = None
    portfolio_url: Optional[str] = None

class Experience(BaseModel):
    company: str
    position: str
    start_date: str  # Can use date type for stricter validation
    end_date: str    # e.g., "Present" or "2023-01-01"
    highlights: list[str]

class Education(BaseModel):
    degree: str
    university: str
    graduation_year: int

class TechnicalSkills(BaseModel):
    languages: list[str]
    frameworks: list[str]
    tools: list[str]

class ResumeSchema(BaseModel):
    personal_info: ContactInfo
    professional_summary: str
    core_competencies: list[str]
    technical_skills: TechnicalSkills
    work_experience: list[Experience]
    education: list[Education]
    certifications: Optional[list[str]] = None



In [ ]:
from ollama import chat
from ollama import ChatResponse
from collections.abc import Iterator
from pydantic import BaseModel

class Text(BaseModel):
    x: int
    y: int
    text: str

class Line(BaseModel):
    x1: int
    y1: int
    x2: int
    y2: int


class Schema(BaseModel):
    text: list[Text]
    lines: list[Line]

class TestSchema(BaseModel):
  name_applicant: str
  profession: str
  skillz: list[str]
print("Started")
pdf_prompt = '''
**Instructions**
You are AI Assistant, you will be given the profile data.
Your task is to transform the given unstructured profile data into a structured format.
1. Understand the given profile data by the user.
2. Create a structured format of json with proper keys of the values.
3. Don't overthink, just take the profile info from the user then make a json structured format.
4. Don't be concerned about the details of how the input was given and how should be the output. It's just basic transformation.
5. The response should be real quick, you shouldn't be thinking too much about the details.
6. Kindly don't think saying But wait the user said so and so. Just output it.
7. Don't be concerned about little details, letter cases and typos.
8. Just provide the output without overthinking and provide the values as it is without overthinking.
'''
pdf_prompt2 = '''
**Instructions**
You are AI Assistant, you will be given the profile data then just simply output in format.
1. Don't worry about cases, typos, expectations of users, no coding or splitting.
2. Don't do any thinking at all just output the data in format.
3. Just provide the output without thinking about anything.
4. If there's any irrelevant lines in the profile data, kindly ignore that without overthinking.
'''
# profile_prompt = '''Context: Hi, My name is Bilal I am an AI/ML Developer, my skills are Langchain, Langraph, Vectordb, RAG systems etc.
# '''
profile_prompt = '''ALEX J. TAYLOR Senior Full-Stack Developer Location: San Francisco, CA Phone: +1-555-0123 Email: alex.taylor@email.com LinkedIn: linkedin.com/in/alextaylor Portfolio: github.com/alextaylor-dev

PROFESSIONAL SUMMARY Innovative Software Engineer with 8+ years of experience in building scalable web applications. Expert in React, Node.js, and Cloud Infrastructure. Proven track record of reducing system latency by 30% and leading cross-functional teams to deliver high-impact products.

CORE COMPETENCIES Software Architecture, Agile Methodology, Cloud Computing (AWS/Azure), CI/CD Pipelines, Database Management.

TECHNICAL SKILLS Languages: JavaScript, TypeScript, Python, SQL, Go. Frameworks: React, Next.js, Express, Django. Tools: Docker, Kubernetes, Git, Terraform, PostgreSQL.

WORK EXPERIENCE TechNova Solutions | Senior Developer | 2020 - Present

Architected a microservices-based platform serving 1M+ active users.

Mentored a team of 5 junior developers, increasing sprint velocity by 20%.

Implemented automated testing that reduced production bugs by 45%.

DataStream Inc. | Full-Stack Engineer | 2017 - 2020

Developed and maintained responsive UI components using React and Redux.

Optimized API endpoints, resulting in a 200ms reduction in load times.

EDUCATION Bachelor of Science in Computer Science University of California, Berkeley | 2017

CERTIFICATIONS AWS Certified Solutions Architect, Google Professional Cloud Developer

Would you like me to fill in this template with your specific details instead?
'''
response: Iterator[ChatResponse] = chat(model='qwen3-vl:4b', messages=[
  {
    'role': 'system',
    'content': pdf_prompt2,
  },
  {
    'role': 'user',
    'content': profile_prompt
  }
], 
stream=True,
think=False,
format=ResumeSchema.model_json_schema())
# format='json')
in_thinking = False
content = ''
thinking = ''
for chunk in response:
  if chunk.message.thinking:
    if not in_thinking:
      in_thinking = True
      print('Thinking:\n', end='', flush=True)
    print(chunk.message.thinking, end='', flush=True)
    # accumulate the partial thinking 
    thinking += chunk.message.thinking
  elif chunk.message.content:
    if in_thinking:
      in_thinking = False
      print('\n\nAnswer:\n', end='', flush=True)
    print(chunk.message.content, end='', flush=True)
    # accumulate the partial content
    content += chunk.message.content

new_messages = [{ 'role': 'assistant', 'thinking': thinking, 'content': content }]
alarm()
# content = response.message.content

Started
Thinking:
Hmm, the user has shared a detailed professional profile for Alex J. Taylor and wants me to output it in a specific format. Let me analyze the instructions carefully.

The key requirements are: 
- Output exactly as formatted without any thinking
- Ignore irrelevant lines
- Don't worry about cases/typos
- No coding or splitting

Looking at the profile data, I notice it's structured with clear sections: contact info, professional summary, core competencies, technical skills, work experience, education, and certifications. There's also a line at the end asking if they want to fill a template, which seems irrelevant to the output requirements.

I need to extract only the relevant information in the requested format. The user emphasized ignoring irrelevant lines, so I'll skip the template question at the end. 

Important to maintain all the details while being precise:
- Contact info must include all fields (location, phone, email, LinkedIn, portfolio)
- Professional summa

In [ ]:
import json
from ollama import chat
from ollama import ChatResponse
from collections.abc import Iterator
from pydantic import BaseModel
def alarm():
    import nava, time
    from nava import stop
    sound_id = nava.play('../alarm.wav', async_mode=True)
    time.sleep(8)
    stop(sound_id)
profile_json = {

    "personal_info": {
        "full_name": "ALEX J. TAYLOR",
        "job_title": "Senior Full-Stack Developer",
        "email": "alex.taylor@email.com",
        "phone": "+1-555-0123",
        "location": "San Francisco, CA",
        "linkedin_url": "linkedin.com/in/alextaylor",
        "portfolio_url": "github.com/alextaylor-dev"
    },
    "professional_summary": "Innovative Software Engineer with 8+ years of experience in building scalable web applications. Expert in React, Node.js, and Cloud Infrastructure. Proven track record of reducing system latency by 30% and leading cross-functional teams to deliver high-impact products.",
    "core_competencies": [
        "Software Architecture",
        "Agile Methodology",
        "Cloud Computing (AWS/Azure)",
        "CI/CD Pipelines",
        "Database Management"
    ],
    "technical_skills": {
        "languages": [
            "JavaScript",
            "TypeScript",
            "Python",
            "SQL",
            "Go"
        ],
        "frameworks": [
            "React",
            "Next.js",
            "Express",
            "Django"
        ],
        "tools": [
            "Docker",
            "Kubernetes",
            "Git",
            "Terraform",
            "PostgreSQL"
        ]
    },
    "work_experience": [
        {
            "company": "TechNova Solutions",
            "position": "Senior Developer",
            "start_date": "2020",
            "end_date": "Present",
            "highlights": [
                "Architected a microservices-based platform serving 1M+ active users",
                "Mentored a team of 5 junior developers, increasing sprint velocity by 20%",
                "Implemented automated testing that reduced production bugs by 45%"
            ]
        },
        {
            "company": "DataStream Inc.",
            "position": "Full-Stack Engineer",
            "start_date": "2017",
            "end_date": "2020",
            "highlights": [
                "Developed and maintained responsive UI components using React and Redux",
                "Optimized API endpoints, resulting in a 200ms reduction in load times"
            ]
        }
    ],
    "education": [
        {
            "degree": "Bachelor of Science in Computer Science",
            "university": "University of California, Berkeley",
            "graduation_year": 2017
        }
    ],
    "certifications": [
        "AWS Certified Solutions Architect",
        "Google Professional Cloud Developer"
    ]
}
# profile_json = json.loads(content)
# profile_json
coordinates_prompt = '''You are an AI HTML Generation Agent.
Your task is to generate a html & css content of a resume.
You will be given a resume json object.
1. You have to generate a resume using html and css, in an A4 Format.
2. Generate a stylish and professional resume layout suitable for printing on A4 paper.
3. Don't do thinking at all, don't worry about typos, cases, splitting.
4. Use good and beautiful design and styling for the, you are free to use any kind of styling and structuring in layouts.
5. Just output the values as it is without explanation or thinking, make the generation answer to happen Quickly.
6. But all the provided values should be included in the resume.
'''

class ElementSchema(BaseModel):
    text: str
    key: str
    x: float
    y: float

class CoordinateJsonSchema(BaseModel):
    output: list[ElementSchema]

response: Iterator[ChatResponse] = chat(model='qwen3-vl:4b', messages=[
  {
    'role': 'system',
    'content': coordinates_prompt,
  },
  {
    'role': 'user',
    'content': str(profile_json)
  }
], 
stream=True,
think=False,
# format=CoordinateJsonSchema.model_json_schema()
)

in_thinking = False
content = ''
thinking = ''
for chunk in response:
  if chunk.message.thinking:
    if not in_thinking:
      in_thinking = True
      print('Thinking:\n', end='', flush=True)
    print(chunk.message.thinking, end='', flush=True)
    # accumulate the partial thinking 
    thinking += chunk.message.thinking
  elif chunk.message.content:
    if in_thinking:
      in_thinking = False
      alarm()
      print('\n\nAnswer:\n', end='', flush=True)
    print(chunk.message.content, end='', flush=True)
    # accumulate the partial content
    content += chunk.message.content

alarm()

Thinking:
We are going to create a professional resume in HTML and CSS for A4 size.
 The design should be clean, modern, and suitable for printing.
 We'll use a two-column layout for the header and content, but note: A4 is 210mm x 297mm.
 However, in practice, we often design for a portrait layout on A4.

 Steps:
 1. Set up the HTML structure with a container for A4 paper.
 2. Create a header with the personal info (full_name, job_title, contact details).
 3. Use sections for: professional summary, core competencies, technical skills, work experience, education, and certifications.
 4. Style with CSS: use a clean font (like Arial, sans-serif), good contrast, and professional color scheme (e.g., dark blue or gray for text on white background).
 5. For A4, we might set the page size and margins appropriately.

 Important: We are to output only the HTML and CSS, without any extra text.

 Let's design:
   - The resume should have a header at the top with name and title.
   - Below that, co

In [26]:
def output_html(content: str, file_name: str):
    from pathlib import Path
    html_content = content.replace('\n', ' ')
    file_name = file_name + '.html'

    path = Path(f'output/html/')
    if not path.exists():
        path.mkdir(exist_ok=True, parents=True)
    path = Path(f'output/html/{file_name}')
    path.write_text(html_content)
output_html(content, 'test3')

In [9]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

canvas = canvas.Canvas("form.pdf", pagesize=letter)
canvas.setLineWidth(.3)
canvas.setFont('Times-Roman', 12)

canvas.drawString(30,750,'OFFICIAL COMMUNIQUE')
canvas.drawString(30,735,'OF ACME INDUSTRIES')
canvas.drawString(500,750,"12/12/2010")
canvas.line(480,747,580,747)

canvas.drawString(275,725,'AMOUNT OWED:')
canvas.drawString(500,725,"$1,000.00")
canvas.line(378,723,580,723)

canvas.drawString(30,703,'RECEIVED BY:')
canvas.line(120,700,580,700)
canvas.drawString(120,703,"JOHN DOE")

canvas.save()